# 02 · Segmentation and density counting (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`), each followed by a check.
Solutions: `../solutions/guided/02_segmentation_density.md`.

**Traces to** `notes/ioai-task-patterns.md` P8.
- T25-RADAR: multi-channel heatmaps, labels −1..3 shifted to 0..4, logits `(B, C, H, W)` vs targets `(B, H, W)`,
  and a metric that weights object pixels ×50.
- T25-H-WEATHER: Dice.
- T25-CHICKEN: density maps whose **sum is the count**, which must be **≥ 0**, at a downsampled resolution.

Syllabus: segmentation (U-Net). Data here is synthetic so the notebook runs anywhere in about 1.5 minutes on a T4.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

def make_maps(n, seed, H=32, W=64):
    """3-channel 'sensor' maps; label -1 = background, 0 = type A blob, 1 = type B blob."""
    rng = np.random.default_rng(seed); yy, xx = np.mgrid[:H, :W]
    X = rng.gamma(2.0, 0.1, (n, 3, H, W)).astype(np.float32); Y = -np.ones((n, H, W), np.int64)
    sig = np.array([[1.0, 0.2, 0.6], [0.3, 1.0, 0.5]])
    for i in range(n):
        for _ in range(rng.integers(1, 4)):
            c = rng.integers(0, 2); cy, cx = rng.uniform(3, H - 3), rng.uniform(3, W - 3)
            blob = ((yy - cy) / rng.uniform(1.5, 3)) ** 2 + ((xx - cx) / rng.uniform(1.5, 4)) ** 2 <= 1
            X[i][:, blob] += (rng.uniform(0.4, 1.0) * sig[c])[:, None]; Y[i][blob] = c
    return X, Y

X_tr, Y_tr = make_maps(1200, 0); X_va, Y_va = make_maps(300, 1)
print(X_tr.shape, Y_tr.shape, "background fraction:", (Y_tr == -1).mean().round(3))

## 1. Dataset: shift labels to 0..C−1

`CrossEntropyLoss` needs class indices in `[0, C)`. T25-RADAR's loader did `labels = labels + 1` and the
inference code subtracted 1 again.

In [ ]:
class Maps(Dataset):
    def __init__(self, X, Y): self.X, self.Y = torch.from_numpy(X), torch.from_numpy(Y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        # TODO(g1): return the image and the label map shifted from -1..1 to 0..2 (still int64)
        return self.X[i], ____

train_loader = DataLoader(Maps(X_tr, Y_tr), batch_size=32, shuffle=True)
val_loader = DataLoader(Maps(X_va, Y_va), batch_size=100)

In [ ]:
xb, yb = next(iter(train_loader))
assert yb.dtype == torch.int64 and yb.min() == 0 and yb.max() == 2 and yb.shape == (32, 32, 64)
print("g1 ok")

## 2. A small U-Net

In [ ]:
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU())

class UNet(nn.Module):
    def __init__(self, cin, n_classes, w=32):
        super().__init__()
        self.e1, self.e2, self.mid = block(cin, w), block(w, 2 * w), block(2 * w, 4 * w)
        # TODO(g2): learnable 2x upsampling from 4w to 2w channels
        self.up2 = nn.ConvTranspose2d(4 * w, 2 * w, ____)
        self.d2 = block(4 * w, 2 * w)
        self.up1 = nn.ConvTranspose2d(2 * w, w, kernel_size=2, stride=2)
        self.d1 = block(2 * w, w)
        self.out = nn.Conv2d(w, n_classes, 1)
    def forward(self, x):
        e1 = self.e1(x)                        # (B, w, H, W)
        e2 = self.e2(F.max_pool2d(e1, 2))      # (B, 2w, H/2, W/2)
        m = self.mid(F.max_pool2d(e2, 2))      # (B, 4w, H/4, W/4)
        # TODO(g3): skip connection: concatenate the upsampled map with e2 along the CHANNEL axis
        d2 = self.d2(torch.cat(____))
        d1 = self.d1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)                    # (B, n_classes, H, W) logits

model = UNet(3, 3).to(device)

In [ ]:
assert model(xb.to(device)).shape == (32, 3, 32, 64)
print("g2, g3 ok")

## 3. Weighted loss that mirrors the weighted metric

T25-RADAR scored background pixels ×1 and object pixels ×50. Train with the same weights so that the loss agrees
with the metric.

In [ ]:
BONUS = 50.0
# TODO(g4): per-pixel cross-entropy with class weights [1, BONUS, BONUS] (class 0 = background after the shift)
loss_fn = nn.CrossEntropyLoss(____)

def radar_score(pred, target, bonus=BONUS):
    """pred/target in ORIGINAL labels (-1 = background)."""
    correct, bg = pred == target, target == -1
    # TODO(g5): weighted correct pixels divided by the maximum achievable weighted score
    return ____

In [ ]:
gt = torch.tensor([[-1, -1, -1], [1, 0, 1], [-1, -1, -1]])
pr = torch.tensor([[-1, 1, -1], [-1, 0, -1], [-1, 1, -1]])
assert abs(float(radar_score(pr, gt)) - (4 + 50) / (6 + 150)) < 1e-6, "compare with the worked example in T25-RADAR"
assert loss_fn.weight.tolist() == [1.0, 50.0, 50.0]
print("g4, g5 ok")

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
for epoch in range(8):
    model.train()
    for xb, yb in train_loader:
        loss = loss_fn(model(xb.to(device)), yb.to(device))
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        scores = []
        for xv, yv in val_loader:
            # TODO(g6): class index per pixel from the logits, then shift back to -1..1
            pred = ____
            scores.append(float(radar_score(pred, yv - 1)))
    print(f"epoch {epoch}  loss {loss.item():.3f}  val radar-score {np.mean(scores):.4f}")

In [ ]:
assert pred.shape == (100, 32, 64) and pred.min() >= -1 and pred.max() <= 1
assert np.mean(scores) > 0.8
print("g6 ok")

## 4. Density maps: counting by integration (T25-CHICKEN)

Each object contributes a small Gaussian whose integral is exactly 1, so `density.sum()` is the count.
Predictions are made at ¼ resolution (as T25-CHICKEN's 180×320 maps for 720×1280 images). To downsample a
density map **and keep its sum**, sum-pool it (average-pool, then multiply by the pooled area).

In [ ]:
def make_counts(n, seed, H=64, W=96, s=1.5):
    rng = np.random.default_rng(seed); yy, xx = np.mgrid[:H, :W]
    img = rng.normal(0.2, 0.05, (n, 1, H, W)).astype(np.float32); den = np.zeros((n, 1, H, W), np.float32)
    for i in range(n):
        for _ in range(rng.poisson(12)):
            cy, cx = rng.uniform(2, H - 2), rng.uniform(2, W - 2)
            g = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * s * s))
            img[i, 0] += 0.8 * g; den[i, 0] += g / g.sum()
    return torch.from_numpy(img), torch.from_numpy(den)

img_tr, den_tr = make_counts(800, 2); img_va, den_va = make_counts(200, 3)
# TODO(g7): downsample the density 4x in each direction WITHOUT changing its total (sum-pooling)
den_tr4 = ____
den_va4 = F.avg_pool2d(den_va, 4) * 16

In [ ]:
assert den_tr4.shape == (800, 1, 16, 24)
assert torch.allclose(den_tr4.sum((1, 2, 3)), den_tr.sum((1, 2, 3)), atol=1e-4), "the count changed"
print("g7 ok, true counts:", den_tr.sum((1, 2, 3))[:5].round().tolist())

In [ ]:
class CountNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(64, 64, 3, padding=2, dilation=2), nn.ReLU(), nn.Conv2d(64, 1, 1))
    def forward(self, x):
        # TODO(g8): densities must be non-negative (T25-CHICKEN scored 0 otherwise): use a smooth positive activation
        return ____

cnet = CountNet().to(device)
opt = torch.optim.Adam(cnet.parameters(), lr=1e-3)
SCALE = 100.0                                 # T25-CHICKEN scaled targets by 100 so the loss is not tiny
for step in range(600):
    idx = torch.randint(0, 800, (32,))
    loss = F.mse_loss(cnet(img_tr[idx].to(device)) * 1.0, den_tr4[idx].to(device) * SCALE)
    opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    pc = (cnet(img_va.to(device)) / SCALE).sum((1, 2, 3)).cpu()
tc = den_va.sum((1, 2, 3))
rel = ((pc - tc).abs() / tc).mean().item()
print(f"mean relative count error {rel:.3f}  ->  T25-CHICKEN score exp(-err) = {np.exp(-rel):.3f}")

In [ ]:
assert (cnet(img_va[:4].to(device)) >= 0).all()
assert rel < 0.2
print("g8 ok, all gaps done")